# Module `validators.py`

Ce notebook explique les validations appliquees aux instances et aux etats dynamiques. Ces validations sont separees pour eviter que `GraphGenerator` devienne responsable de toute la logique du projet.

## Deux niveaux de validation

`InstanceValidator` valide le graphe statique genere : densite de base, densite residuelle, degre moyen residuel et connexite.

`DynamicStateValidator` valide un etat dynamique : on refuse de passer une route en `OFF` si cela casse la connexite, rend le reseau trop peu dense, fait tomber le degre moyen actif trop bas ou depasse le ratio maximal de routes indisponibles.

In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent / "src"))

from cesipath.graph_generator import GraphGenerator
from cesipath.models import GraphGenerationConfig
from cesipath.validators import DynamicStateValidator, InstanceValidator


In [ ]:
config = GraphGenerationConfig(node_count=8, seed=42)
generator = GraphGenerator(config)
instance = generator.generate()

validator = InstanceValidator(config)
print("Instance valide ?", validator.is_valid(instance))
print(instance.summary())


## Validation dynamique

La dynamique ne coupe pas une route aveuglement. Avant de retirer une arete, on simule son retrait et on demande au validateur si l'etat resterait acceptable.

In [ ]:
snapshot = generator.initialize_dynamic_snapshot(instance)
dynamic_validator = DynamicStateValidator(instance)
candidate = next(iter(snapshot.edge_availability))

print("Arete candidate :", candidate)
print("Peut-on la passer OFF ?", dynamic_validator.can_disable_edge(snapshot.edge_availability, candidate))
